# Explorative Data Analysis (EDA) of the ANZ Synthesis Transactional Dataset

In [3]:
# Importing Required Libraries
import numpy as np
import pandas as pd 
import plotly.express as px
import warnings
import math
import plotly.graph_objects as go
import plotly.express as px
import plotly.figure_factory as ff
from plotly.subplots import make_subplots
import plotly.io as pio

warnings.filterwarnings('ignore')

## 1. Data Importing

In [4]:
# 1. Importing the ANZ Synthesised Transaction Dataset
df_raw = pd.read_csv('anz_internship_raw_data.csv')
df_raw.head()   # Show the top 5 rows

,status,card_present_flag,bpay_biller_code,account,currency,long_lat,txn_description,merchant_id,merchant_code,first_name,...,age,merchant_suburb,merchant_state,extraction,amount,transaction_id,country,customer_id,merchant_long_lat,movement
0,authorized,1.0,NaN,ACC-1598451071,AUD,153.41 -27.95,POS,81c48296-73be-44a7-befa-d053f48ce7cd,NaN,Diana,...,26,Ashmore,QLD,2018-08-01T01:01:15.000+0000,16.25,a623070bfead4541a6b0fff8a09e706c,Australia,CUS-2487424745,153.38 -27.99,debit
1,authorized,0.0,NaN,ACC-1598451071,AUD,153.41 -27.95,SALES-POS,830a451c-316e-4a6a-bf25-e37caedca49e,NaN,Diana,...,26,Sydney,NSW,2018-08-01T01:13:45.000+0000,14.19,13270a2a902145da9db4c951e04b51b9,Australia,CUS-2487424745,151.21 -33.87,debit
2,authorized,1.0,NaN,ACC-1222300524,AUD,151.23 -33.94,POS,835c231d-8cdf-4e96-859d-e9d571760cf0,NaN,Michael,...,38,Sydney,NSW,2018-08-01T01:26:15.000+0000,6.42,feb79e7ecd7048a5a36ec889d1a94270,Australia,CUS-2142601169,151.21 -33.87,debit
3,authorized,1.0,NaN,ACC-1037050564,AUD,153.10 -27.66,SALES-POS,48514682-c78a-4a88-b0da-2d6302e64673,NaN,Rhonda,...,40,Buderim,QLD,2018-08-01T01:38:45.000+0000,40.90,2698170da3704fd981b15e64a006079e,Australia,CUS-1614226872,153.05 -26.68,debit
4,authorized,1.0,NaN,ACC-1598451071,AUD,153.41 -27.95,SALES-POS,b4e02c10-0852-4273-b8fd-7b3395e32eb0,NaN,Diana,...,26,Mermaid Beach,QLD,2018-08-01T01:51:15.000+0000,3.25,329adf79878c4cf0aeb4188b4691c266,Australia,CUS-2487424745,153.44 -28.06,debit


In [5]:
df_raw.shape # The data contains 12043 rows and 23 columns

df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12043 entries, 0 to 12042
Data columns (total 23 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   status             12043 non-null  object 
 1   card_present_flag  7717 non-null   float64
 2   bpay_biller_code   885 non-null    object 
 3   account            12043 non-null  object 
 4   currency           12043 non-null  object 
 5   long_lat           12043 non-null  object 
 6   txn_description    12043 non-null  object 
 7   merchant_id        7717 non-null   object 
 8   merchant_code      883 non-null    float64
 9   first_name         12043 non-null  object 
 10  balance            12043 non-null  float64
 11  date               12043 non-null  object 
 12  gender             12043 non-null  object 
 13  age                12043 non-null  int64  
 14  merchant_suburb    7717 non-null   object 
 15  merchant_state     7717 non-null   object 
 16  extraction         120

## 2. Data Cleaning

In [6]:
# 'bpay_biller_code' & 'merchant_code' have the majority of null values, so the rows of missing data must be dropped

df_raw.isnull().sum() # showing all rows that contain any columns with null values

df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12043 entries, 0 to 12042
Data columns (total 23 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   status             12043 non-null  object 
 1   card_present_flag  7717 non-null   float64
 2   bpay_biller_code   885 non-null    object 
 3   account            12043 non-null  object 
 4   currency           12043 non-null  object 
 5   long_lat           12043 non-null  object 
 6   txn_description    12043 non-null  object 
 7   merchant_id        7717 non-null   object 
 8   merchant_code      883 non-null    float64
 9   first_name         12043 non-null  object 
 10  balance            12043 non-null  float64
 11  date               12043 non-null  object 
 12  gender             12043 non-null  object 
 13  age                12043 non-null  int64  
 14  merchant_suburb    7717 non-null   object 
 15  merchant_state     7717 non-null   object 
 16  extraction         120

In [7]:
# Suspected redundant columns
list_names = ['bpay_biller_code', 'card_present_flag', 'merchant_code', 'merchant_suburb', 'merchant_state']

for element in list_names:
    print(f"{element}: {df_raw[element].unique()}")

# Summary
# The 'bpay_biller_code' and 'merchant_code' do not have consistent values - So they are to be removed
columns_to_remove = ['bpay_biller_code', 'merchant_code']

bpay_biller_code: [nan '0' ' THE DISCOUNT CHEMIST GROUP'
 ' LAND WATER & PLANNING East Melbourne']
card_present_flag: [ 1.  0. nan]
merchant_code: [nan  0.]
merchant_suburb: ['Ashmore' 'Sydney' 'Buderim' ... 'Donvale' 'Ulmarra' 'Kings Park']
merchant_state: ['QLD' 'NSW' nan 'VIC' 'WA' 'SA' 'NT' 'TAS' 'ACT']


In [8]:
# Checking for unique values of each column

df = df_raw.copy()
one_count_columns = []

for col in df_raw.columns:
    
    # drop column if the count is 1 or in columns_to_remove list
    if (len(df_raw[col].unique()) == 1) or (col in columns_to_remove):
        one_count_columns.append(col)

df.drop(one_count_columns, axis=1, inplace=True)

In [9]:
# Chicking if there are any duplicate records in the dataset
if df.duplicated().sum() == 0:
    print("There are no duplicates.")
    df_cleaned = df.copy()  # Copying the cleaned dataset
    
else:
    print("There are duplicates in the dataset.")

There are no duplicates.


## 3. Data Exploration and Analysis

In [9]:
# Separate the data by age to analyse data per age group
df_cleaned['age_group'] = pd.cut(df_cleaned['age'], [0, 20, 30, 40, 50, 60, max(df_cleaned['age'])], 
                            labels=['<20', '20-30', '30-40', '40-50', '50-60', '>60'])

df_cleaned.head(2)

,status,card_present_flag,account,long_lat,txn_description,merchant_id,first_name,balance,date,gender,age,merchant_suburb,merchant_state,extraction,amount,transaction_id,customer_id,merchant_long_lat,movement,age_group
0,authorized,1.0,ACC-1598451071,153.41 -27.95,POS,81c48296-73be-44a7-befa-d053f48ce7cd,Diana,35.39,8/1/2018,F,26,Ashmore,QLD,2018-08-01T01:01:15.000+0000,16.25,a623070bfead4541a6b0fff8a09e706c,CUS-2487424745,153.38 -27.99,debit,20-30
1,authorized,0.0,ACC-1598451071,153.41 -27.95,SALES-POS,830a451c-316e-4a6a-bf25-e37caedca49e,Diana,21.20,8/1/2018,F,26,Sydney,NSW,2018-08-01T01:13:45.000+0000,14.19,13270a2a902145da9db4c951e04b51b9,CUS-2487424745,151.21 -33.87,debit,20-30


In [10]:
# Convert the datatype of 'date' and 'extraction' columns to datetime datatype
df_cleaned.loc[:, ['date', 'extraction']] = df_cleaned.loc[:, ['date', 'extraction']].apply(pd.to_datetime, errors='coerce')

df_cleaned.head(2) # Confirm if the datatype of the 'date' and 'extraction' columns have been updated

,status,card_present_flag,account,long_lat,txn_description,merchant_id,first_name,balance,date,gender,age,merchant_suburb,merchant_state,extraction,amount,transaction_id,customer_id,merchant_long_lat,movement,age_group
0,authorized,1.0,ACC-1598451071,153.41 -27.95,POS,81c48296-73be-44a7-befa-d053f48ce7cd,Diana,35.39,2018-08-01 00:00:00,F,26,Ashmore,QLD,2018-08-01 01:01:15+00:00,16.25,a623070bfead4541a6b0fff8a09e706c,CUS-2487424745,153.38 -27.99,debit,20-30
1,authorized,0.0,ACC-1598451071,153.41 -27.95,SALES-POS,830a451c-316e-4a6a-bf25-e37caedca49e,Diana,21.20,2018-08-01 00:00:00,F,26,Sydney,NSW,2018-08-01 01:13:45+00:00,14.19,13270a2a902145da9db4c951e04b51b9,CUS-2487424745,151.21 -33.87,debit,20-30


In [11]:
df_cleaned.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12043 entries, 0 to 12042
Data columns (total 20 columns):
 #   Column             Non-Null Count  Dtype   
---  ------             --------------  -----   
 0   status             12043 non-null  object  
 1   card_present_flag  7717 non-null   float64 
 2   account            12043 non-null  object  
 3   long_lat           12043 non-null  object  
 4   txn_description    12043 non-null  object  
 5   merchant_id        7717 non-null   object  
 6   first_name         12043 non-null  object  
 7   balance            12043 non-null  float64 
 8   date               12043 non-null  object  
 9   gender             12043 non-null  object  
 10  age                12043 non-null  int64   
 11  merchant_suburb    7717 non-null   object  
 12  merchant_state     7717 non-null   object  
 13  extraction         12043 non-null  object  
 14  amount             12043 non-null  float64 
 15  transaction_id     12043 non-null  object  
 16  cust

In [12]:
# Summary: The 'date' and 'extraction' are object datatype, they must be datetime
df_cleaned['date'] = pd.to_datetime(df_cleaned['date'], errors ='coerce')
df_cleaned['extraction'] = pd.to_datetime(df_cleaned['extraction'], errors ='coerce')

# Create date helper columns
df_cleaned['month'] = df_cleaned['date'].dt.month_name()
df_cleaned['day'] = df_cleaned['date'].dt.day_name()
df_cleaned['hour']= df_cleaned['extraction'].dt.hour
df_cleaned.head()

# Change datatype of card_present_flag to INT
df.card_present_flag = df.card_present_flag.astype('Int64')
df.head()

df_cleaned.info() # Check datatypes

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12043 entries, 0 to 12042
Data columns (total 23 columns):
 #   Column             Non-Null Count  Dtype              
---  ------             --------------  -----              
 0   status             12043 non-null  object             
 1   card_present_flag  7717 non-null   float64            
 2   account            12043 non-null  object             
 3   long_lat           12043 non-null  object             
 4   txn_description    12043 non-null  object             
 5   merchant_id        7717 non-null   object             
 6   first_name         12043 non-null  object             
 7   balance            12043 non-null  float64            
 8   date               12043 non-null  datetime64[ns]     
 9   gender             12043 non-null  object             
 10  age                12043 non-null  int64              
 11  merchant_suburb    7717 non-null   object             
 12  merchant_state     7717 non-null   object     

In [13]:
# Key Observation

# The Non-Null Count is not the same for all columns
df_cleaned.isnull().sum() # showing all rows that contain any columns with null values

status                  0
card_present_flag    4326
account                 0
long_lat                0
txn_description         0
merchant_id          4326
first_name              0
balance                 0
date                    0
gender                  0
age                     0
merchant_suburb      4326
merchant_state       4326
extraction              0
amount                  0
transaction_id          0
customer_id             0
merchant_long_lat    4326
movement                0
age_group               0
month                   0
day                     0
hour                    0
dtype: int64

In [10]:
# Check the number of columns with null values
df_cleaned.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12043 entries, 0 to 12042
Data columns (total 19 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   status             12043 non-null  object 
 1   card_present_flag  7717 non-null   float64
 2   account            12043 non-null  object 
 3   long_lat           12043 non-null  object 
 4   txn_description    12043 non-null  object 
 5   merchant_id        7717 non-null   object 
 6   first_name         12043 non-null  object 
 7   balance            12043 non-null  float64
 8   date               12043 non-null  object 
 9   gender             12043 non-null  object 
 10  age                12043 non-null  int64  
 11  merchant_suburb    7717 non-null   object 
 12  merchant_state     7717 non-null   object 
 13  extraction         12043 non-null  object 
 14  amount             12043 non-null  float64
 15  transaction_id     12043 non-null  object 
 16  customer_id        120

## 4. Analysis of Categorical variables

In [15]:
# Metadata of the dataset
# status - denotes the status of the transaction

# card_present_flag - did the customer use card for the transaction. 1 for yes and 0 for no

# bpay_biller_code - unique code of the BPay Transaction done by the customer

# account - customer's account number

# currency - Australian Dollar

# long_lat - transaction's location

# txn_description - mode of transaction. POS or SALES-POS

# merchant_id - merchant's id

# merchant_code - unique merchant code for each customers

# first_name - customer's first name

In [16]:

# Configuration and Data Setup
columns_of_interest = ['status', 'card_present_flag', 'txn_description' , 'movement' , 'gender', 'merchant_state']
column_names = ['Transaction Status', 'Card Present Flag', 'Mode of Transaction' , 'Movement', 'Gender', 'Merchant State']

plot_title = "Analysis of Categorical Variables (Frequency / Percentage)"

# Subplot initialization - Number of rows and columns for the sublots
cols = 2
rows = math.ceil(len(columns_of_interest) / cols)

# Cleaned-up subplot titles
subplot_titles = column_names

fig = make_subplots(
    rows=rows, 
    cols=cols,
    subplot_titles=subplot_titles,
    horizontal_spacing=0.15, 
    vertical_spacing=0.12, shared_yaxes=True)

# Populate Subplots Dynamically
for index, col_name in enumerate(columns_of_interest):
    # Calculate row and column positions
    row = (index // cols) + 1
    col = (index % cols) + 1
    
    # Compute counts once to optimize performance
    counts = df_cleaned[col_name].value_counts()
    percentages = df_cleaned[col_name].value_counts(normalize=True) * 100
    
    # Generate text labels (e.g., "42 (15%)")
    text_labels = [f"{count}<br>({pct:.1f}%)" for count, pct in zip(counts, percentages)]
    
    fig.add_trace(go.Bar(x=counts.index, y=counts, name=col_name,
            text=text_labels, textposition='auto',
            hovertemplate="<b>Category:</b> %{x}<br><b>Count:</b> %{y}<extra></extra>"), 
            row=row, col=col)

# Global Layout and Styling Customization
fig.update_layout(title=dict(text=plot_title, 
        x=0.5, y=0.98, 
        xanchor='center', yanchor='top'), 
        title_font_size=22, 
        showlegend=False, 
        height=300 * rows,  # Scales perfectly with the number of rows
        margin=dict(l=50, r=50, t=100, b=50),
        template="plotly_white",  # Clean, modern aesthetic
        )

# Links all y-axes to scale together when you zoom or pan
fig.update_yaxes(title_text="Number of transactions")

fig.show()

In [17]:
print((6285-5758)/6285*100)

8.385043754972155


In [18]:
# Key Observations:

# About 64.1% of transactions were authorized and the 35.9% were posted.
# The majority (64.1%) of the transactions were done using "SALES-POS" & "POS"
# Males tend to do more transactions as compared to females, with 8.4% more males than females
# More than 80% of transactions were done using a physical card from a customer
# 92.7% transactions were conducted using a debit and the rest using a credit.
# NSW, VIC, QLD are the most busy merchant states, where as ACT and TAS are the least busy.

In [19]:
df_cleaned.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12043 entries, 0 to 12042
Data columns (total 23 columns):
 #   Column             Non-Null Count  Dtype              
---  ------             --------------  -----              
 0   status             12043 non-null  object             
 1   card_present_flag  7717 non-null   float64            
 2   account            12043 non-null  object             
 3   long_lat           12043 non-null  object             
 4   txn_description    12043 non-null  object             
 5   merchant_id        7717 non-null   object             
 6   first_name         12043 non-null  object             
 7   balance            12043 non-null  float64            
 8   date               12043 non-null  datetime64[ns]     
 9   gender             12043 non-null  object             
 10  age                12043 non-null  int64              
 11  merchant_suburb    7717 non-null   object             
 12  merchant_state     7717 non-null   object     

In [20]:
# Analysis the mode of transactions

df_grp_transaction_modes = df_cleaned.groupby(by='txn_description')['amount'].sum().round().reset_index()

# Generate the treemap
fig = px.treemap(df_grp_transaction_modes, path=['txn_description'], values='amount',
    color='amount', color_continuous_scale='Viridis'
)

# Polish the layout
fig.update_layout(
    title=dict(text="<b>Total Amount in Australian Dollar by Transaction Description</b>",
        x = 0.5,y = 0.95, xanchor = 'center'),
    margin=dict(l=10, r=10, t=70, b=10),
    coloraxis_showscale=False)

# Configure text display inside the boxes
fig.data[0].textinfo = 'label+value'

fig.show()

In [21]:
# Key Observations:

# Pay/Salary is the major contributor of bank transactional amounts. 
# This is expected as salary transaction amount is usually very high as compared to normal debit transactions.

In [22]:
# Analysis the total amount transacted by surburb
df_grp_amount_by_surburb = df_cleaned.groupby(by='merchant_suburb')['amount'].sum().round().reset_index()

# Generate the treemap
fig = px.treemap(df_grp_amount_by_surburb, path=['merchant_suburb'],
           values='amount', color = 'amount',)

# Polish the layout
fig.update_layout(title=dict(text = "Total Transactional Amount by Suburb", x = 0.5, y = 0.95),
                    margin=dict(l=10, r=10, t=50, b=10), showlegend=False,)

# Configure text display inside the boxes
fig.data[0].textinfo = 'label+value'
fig.update_traces(marker_coloraxis = None)
fig.show()

In [23]:
# Key Observation:

# Sydney, Melbourne, South Brisbane, and Mascot are the leading surburbs by the total value of transaction amount.

In [24]:
# Analyzing the debit transactions
df_debit_transactions = df_cleaned[df_cleaned['movement'] == 'debit'] # Debit Transactions
df_debit_transactions.shape # 11160 debit transactions

# Configuration and Data Setup
column_names = ['status', 'card_present_flag', 'txn_description', 'movement', 'gender', 'merchant_state']
plot_title = "Total Transactional Amount in Australian Dollar and Percentage"


# Dynamic grid generation
coloumns_per_row = 2
rows = math.ceil(len(column_names) / coloumns_per_row)

# Clean, professional subplot titles
subplot_titles = ['Transaction Status', 'Card Present Flag', 'Mode of Transaction' , 'Movement', 'Gender', 'Merchant State']

fig = make_subplots(
    rows=rows, 
    cols=coloumns_per_row,
    subplot_titles=subplot_titles,
    horizontal_spacing=0.15, 
    vertical_spacing=0.15
)

# 2. Populate Subplots Dynamically
for index, col_name in enumerate(column_names):
    # Calculate exact subplot positioning
    row = (index // coloumns_per_row) + 1
    col = (index % coloumns_per_row) + 1
    
    # CRITICAL FIX: Group, aggregate, and sum ONCE per column to maximize performance
    grouped = df_debit_transactions.groupby(by=col_name)['amount'].sum().reset_index()
    
    # Calculate percentages accurately using the aggregated total
    total_amount = grouped['amount'].sum()
    
    # Create descriptive text labels (e.g., "$1,250\n(15.4%)")
    text_labels = [
        f"${amt:,.0f}<br>({(amt / total_amount) * 100:.1f}%)" 
        if total_amount > 0 else "$0<br>(0%)"
        for amt in grouped['amount']
    ]
    
    fig.add_trace(
        go.Bar(
            x=grouped[col_name],                      # FIX: Passed the category series
            y=grouped['amount'].round(2),             # FIX: Passed the numeric series
            name=col_name,
            text=text_labels,
            textposition='auto',
            hovertemplate="<b>Category:</b> %{x}<br><b>Total Amount:</b> $%{y:,.2f}<extra></extra>"
        ),
        row=row, 
        col=col
    )

fig.update_layout(
    title=dict(
        text=plot_title, 
        x=0.5, 
        y=0.98, 
        xanchor='center', 
        yanchor='top'
    ),
    title_font_size=22, 
    showlegend=False, 
    height=320 * rows,  # Scales perfectly regardless of how many columns you analyze
    margin=dict(l=60, r=60, t=120, b=60),
    template="plotly_white"  # Clean, modern layout look
)

# Global Layout and Styling Customization
fig.update_layout(title=dict(text=plot_title, 
        x=0.5, y=0.98, 
        xanchor='center', yanchor='top'), 
        title_font_size=22, 
        showlegend=False, 
        height=300 * rows,  # Scales perfectly with the number of rows
        margin=dict(l=50, r=50, t=100, b=50),
        template="plotly_white",  # Clean, modern aesthetic
        )

# Links all y-axes to scale together when you zoom or pan
fig.update_yaxes(title_text="Value of transactions")

fig.show()

In [25]:
# Key Obervations:

# 80% of the amount was transacted using cards.
# Payment mode of transaction contributes most to the transaction amount.
# NSW & VIC merchant states contributed more than half to overall transaction amount

In [26]:
df_grp_gender_by_state = df_cleaned.groupby(by=['merchant_state', 'gender'])[['amount']].sum().reset_index()
order = df_cleaned.groupby(by=['merchant_state'])[['amount']].sum().sort_values(by='amount',ascending=False).index
df_grp_gender_by_state['merchant_state'] = pd.Categorical(df_grp_gender_by_state['merchant_state'],order)
df_grp_gender_by_state = df_grp_gender_by_state.groupby(by=['merchant_state', 'gender']).sum().reset_index()

fig = px.bar(data_frame=df_grp_gender_by_state,
       x='merchant_state', y='amount', color='gender',
       barmode='group', text=df_grp_gender_by_state.amount.apply(lambda x : str(round(x/1000,2))+'k'))

fig.update_traces(textposition='outside')
fig.update_xaxes(title='Merchant State') 
fig.update_yaxes(title='Transaction Amount (Australian Dollar)')
fig.update_layout(title=dict(text = "Transaction Amount (Australian Dollar) in Merchant State by Gender", 
                             x=0.5, y=0.95),
                    title_font_size=20,)
fig.show()

In [27]:
# Key Observation:
# NSW and VIC are the primary drivers of transaction volume by a massive margin. 
# Combined, they account for roughly $189k in transactions, which eclipses all other states and territories combined.
# ACT and TAS represent negligible market share, with total transaction volumes lingering below $5k per state.

# In High-Volume Markets (NSW, VIC), Male spending strongly dominates. 
# For instance, in NSW, male transactions ($60.59k) outperform female transactions ($41.43k) by nearly 46%.
# In Mid-to-Low Volume Markets (QLD, WA, SA, NT), Female spending consistently outperforms male spending. 
# In states like Western Australia (WA) and South Australia (SA), women spend roughly double what men do (e.g., $11.35k vs $5.43k in SA).

# The Key Takeaway is that male-targeted campaigns should be primarily around NSW and VIC.
# Focus female-targeted campaigns or localized promotions in the expanding mid-tier states (QLD, WA, SA), where women are clearly driving the local transaction volume.

In [29]:
df_grp_gender_by_state

,merchant_state,gender,amount
0,NSW,F,41430.88
1,NSW,M,60590.89
2,VIC,F,38626.01
3,VIC,M,48957.99
4,QLD,F,28611.05
5,QLD,M,24872.40
6,WA,F,19908.15
7,WA,M,14083.91
8,SA,F,11349.73
9,SA,M,5426.84


In [34]:
import pandas as pd

# Chronological order
days_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

# Convert the 'day' column to a Categorical type with this order
df_cleaned['day'] = pd.Categorical(df_cleaned['day'], categories=days_order, ordered=True)

# Compute value counts and sort by the index (the days)
day_counts = df_cleaned['day'].value_counts().sort_index()

# Pass cleanly to your Plotly chart variables
x = day_counts.index.tolist()
y = day_counts.tolist()



# For number of daily transactions
fig_number_of_transactions = px.bar(data_frame=df_cleaned,
         x = day_counts.index.tolist(),
         y = day_counts.tolist(),
      #  x=df_cleaned['day'].value_counts().index.tolist(), 
      #  y=df_cleaned['day'].value_counts().tolist(), 
       text=df_cleaned['day'].value_counts().tolist())

fig_number_of_transactions.update_traces(textposition='outside', marker_coloraxis=None)
fig_number_of_transactions.update_xaxes(title='Day of the week') 
fig_number_of_transactions.update_yaxes(title='Number of Transactions')
fig_number_of_transactions.update_layout(title=dict(text = "Number of Transactions per Weekday", x=0.5, y=0.95),
                  title_font_size=20, showlegend=False, height = 450,)
fig_number_of_transactions.show()

# For the dollar value of the daily transactions
fig_value_of_transactions_per_day = px.bar(data_frame=df_cleaned.groupby(by='day')[['amount']].sum().sort_values('amount', ascending=False),
            text=df_cleaned.groupby(by='day')[['amount']].sum().sort_values('amount', ascending=False)['amount'].apply(lambda x : str(round(x/1000,2))+'k'))

fig_value_of_transactions_per_day.update_traces(textposition='outside')
fig_value_of_transactions_per_day.update_xaxes(title='Day of the week')
fig_value_of_transactions_per_day.update_yaxes(title='Transaction Amount (Australian Dollar)')
fig_value_of_transactions_per_day.update_layout(title=dict(text = "Transaction Amount (Australian Dollar) per Weekday", x=0.5, y=0.95),
                    title_font_size=20, showlegend=False, height = 450,)
fig_value_of_transactions_per_day.show()

In [ ]:
# Observation:

# The transaction count is lower during the start of the week but start to pick up on wednesday through saturday.
# Even though transaction count is comparatively less on satuday but it is still at place 2 in terms of transaction amount which signifies bigger transactions on Saturday.

In [ ]:
df1_grp=df1.groupby(by=['day','gender'])[['amount']].sum().reset_index()
order = ['Monday','Tuesday', 'Wednesday','Thursday','Friday','Saturday','Sunday']
df1_grp['day']=pd.Categorical(df1_grp['day'],order) 
df1_grp= df1_grp.groupby(by=['day','gender']).sum().reset_index()
fig=px.bar(data_frame=df1_grp,
       x='day',
       y='amount',color='gender',
       barmode='group',
       text=df1_grp.amount.apply(lambda x : str(round(x/1000,2))+'k')
      )
fig.update_traces(textposition='outside')
fig.update_xaxes(title='Day') 
fig.update_yaxes(title='Transaction Amount')
fig.update_layout(
                    title=dict(text = "Transaction Amount per day by Gender",x=0.5,y=0.95),
                    title_font_size=20,
                  )
fig.show()

In [ ]:
# Insights

# Males spent most on Saturday which may be due to dinner date,family dinner or some other weekend plans.
# Females are spending most on Wednesday & Friday.

In [ ]:
fig= px.bar(data_frame=df,
       x=df['month'].value_counts().index.tolist(),
       y=df['month'].value_counts().tolist(),
       color=df['month'].value_counts().tolist(),
       text=df['month'].value_counts().tolist()
      )
fig.update_traces(textposition='outside')
fig.update_xaxes(title='Month') 
fig.update_yaxes(title='Transaction Count')
fig.update_layout(
                    title=dict(text = "Transaction flow by each month",x=0.5,y=0.95),
                    title_font_size=20,
                    width = 700,
                    height = 450,
                  )
fig.show()

In [ ]:
# Insights : As per the above bar graph there is a steady increase in the number of transaction by each passing Month which is a good sign

In [ ]:
fig=px.bar(df.groupby(by='customer_id').sum()['amount'].sort_values(ascending=False).head(10),
       color=df.groupby(by='customer_id').sum()['amount'].sort_values(ascending=False).head(10),
       text=df.groupby(by='customer_id').sum()['amount'].sort_values(ascending=False).head(10).round(),
      )
fig.update_traces(textposition='outside',marker_coloraxis=None)
fig.update_xaxes(title='Customer ID') 
fig.update_yaxes(title='Transaction Amount')
fig.update_layout(
                    title=dict(text = "Top 10 customers by Transaction Amount",x=0.5,y=0.95),
                    title_font_size=20,
                    showlegend=False,
                    height = 500,
                  )
fig.show()

In [ ]:
fig=px.bar(df1.age_group.value_counts(),
       color=df1.age_group.value_counts(),
       text=df1.age_group.value_counts().tolist(),
      )
fig.update_traces(textposition='outside',marker_coloraxis=None)
fig.update_xaxes(title='Age Group') 
fig.update_yaxes(title='Transaction Count')
fig.update_layout(
                    title=dict(text = "Transactions by Age Group",x=0.5,y=0.95),
                    title_font_size=20,
                    showlegend=False,
                    height = 450,
                  )
fig.show()

In [ ]:
# Observation

# Most transactions have been been carried out by Age Groups - "20-30" & "30-40".
# Company should think of providing some attractive offers for "50-60" & ">60" age groups considering the transaction volume of these groups.

In [ ]:
df2_grp=df1.groupby(by=['age_group','gender']).sum()['amount'].reset_index()
fig=px.bar(data_frame=df2_grp,
       x = 'age_group',
       y = 'amount',
       color='gender',
       barmode='group',
       text=df2_grp.amount.apply(lambda x : str(round(x/1000,2))+'k')
      )
fig.update_traces(textposition='outside')
fig.update_layout(
                    title=dict(text = "Transaction Amount by Age Group & Gender",x=0.5,y=0.95),
                    title_font_size=20,
                  )
fig.show()

In [ ]:
# Observation:

# Males in the age group of 20-30 are contributing most to the Total Txn amount.
# In Age group '<20', Females are ahead of males in terms of Total txn amount

In [ ]:
df3_grp=df1.groupby(by='date').mean()[['amount']].merge(df1.groupby(by='date').count()[['transaction_id']],on='date')
df3_grp.columns= ['Amount','Transaction Count']
fig=px.line(df3_grp)
fig.update_xaxes(title='Date') 
fig.update_layout(
                    title=dict(text = "Average Amount VS Txn Count over time",x=0.5,y=0.95),
                    title_font_size=20
                  )
fig.show()

In [ ]:
# Observation:

# The average transaction amount on 7th August & Oct 21st was very high approx 100 AUD.
# Large number of transactions took place on 17th August & 28th September.

In [ ]:
fig=px.line(df1.groupby(by='date')[['amount']].sum())
fig.update_traces(line=dict(color="#8cba51", width=3.5))
fig.update_xaxes(title='Date') 
fig.update_yaxes(title='Transaction Amount')
fig.update_layout(
                    title=dict(text = "Total Txn Amount over time",x=0.5,y=0.95),
                    title_font_size=20,
                    showlegend=False,
                  )
fig.show()

In [ ]:
# Insights: Total Transaction amount almost touched 14k AUD on 21st Oct. Looks like some big transaction were done on that day as the transaction count is not that high on 21st Oct.

In [ ]:
fig=px.line(df1.groupby(by='hour')[['amount']].sum(),
            text=df1.groupby(by='hour').sum()['amount'].apply(lambda x : str(round(x/1000))+'k').values
            )
fig.update_traces(line=dict(color="#f58634", width=5))
fig.update_xaxes(title='Hour') 
fig.update_yaxes(title='Transaction Amount')
fig.update_layout(
                    title=dict(text = "Total Txn Amount hourly",x=0.5,y=0.95),
                    title_font_size=20,
                    showlegend=False,
                  )
fig.update_traces(textposition='middle right',fillcolor='red')
fig.show()

In [ ]:
# Insights:

# Total transaction amount generated at 9:00 AM - 9:59 AM is approx 47k which is highest throughout the day.
# Between 12:00 AM - 7:00 AM we have least transaction amount because of off hours.

In [ ]:
df4_grp= df1.groupby(by=['hour','month','gender']).agg(['count','sum'])[['amount']].reset_index()
df4_grp.columns = ['hour', 'month' ,'gender','Transaction Count', 'Total Txn Amount']
fig1=px.line(data_frame=df4_grp,
            x=df4_grp.hour,
            y=df4_grp['Transaction Count'],
            color=df4_grp.gender,
            facet_col= df4_grp.month
           )
fig1.update_xaxes(title='Hour') 
fig1.update_layout(
                    title=dict(text = "Hourly Transaction count by Month ",x=0.5,y=0.95),
                    title_font_size=20,
                    margin=dict(l=80, r=80, t=100, b=80)
                  )
fig1.show()


fig2=px.line(data_frame=df4_grp,
            x=df4_grp.hour,
            y=df4_grp['Total Txn Amount'],
            color=df4_grp.gender,
            facet_col= df4_grp.month
           )
fig2.update_xaxes(title='Hour') 
fig2.update_layout(
                    title=dict(text = "Hourly Transaction Amount by Month ",x=0.5,y=0.95),
                    title_font_size=20,
                    margin=dict(l=80, r=80, t=100, b=80)
                  )
fig2.show()

In [ ]:
# Insights:

# In the month of September & October even though transaction count by females are more at 9:00 AM but TXN amount is still less. Seems like comparatively small transactions done by females during the start of the day.

# In October at 2:00 PM transaction amount by females is almost double as compared to males.

In [ ]:
df4_grp= df1.groupby(by=['hour','day','gender']).agg(['count','sum'])[['amount']].reset_index()
df4_grp.columns = ['hour', 'day' ,'gender','Transaction Count', 'Total Txn Amount']
fig1=px.line(data_frame=df4_grp,
            x=df4_grp.hour,
            y=df4_grp['Transaction Count'],
            color=df4_grp.gender,
            facet_col= df4_grp.day
           )
fig1.update_xaxes(title='Hour') 
fig1.update_layout(
                    title=dict(text = "Hourly Transaction count by Day ",x=0.5,y=0.95),
                    title_font_size=20,
                    margin=dict(l=80, r=80, t=100, b=80)
                  )
fig1.show()


fig2=px.line(data_frame=df4_grp,
            x=df4_grp.hour,
            y=df4_grp['Total Txn Amount'],
            color=df4_grp.gender,
            facet_col= df4_grp.day
           )
fig2.update_xaxes(title='Hour') 
fig2.update_layout(
                    title=dict(text = "Hourly Transaction Amount by Day ",x=0.5,y=0.95),
                    title_font_size=20,
                    margin=dict(l=80, r=80, t=100, b=80)
                  )
fig2.show()

In [ ]:
# Insights:

# On Saturday between 2:00 PM - 3:00 PM transaction amount by males is almost 6 times higher than females. However on Sunday at the same time the trend is completely in the opposite direction.

In [ ]:
## Analysing Credit Transactions

In [ ]:
df2 = df[df.movement=='credit']
df2.shape # 883 Credit transactions

In [ ]:
fig=px.bar(
            df2.groupby(by='customer_id').mean()['balance'].sort_values(ascending=False).head(10),
            text = df2.groupby(by='customer_id').mean()['balance'].sort_values(ascending=False).head(10).apply(
                lambda x : str(round(x/1000,2))+'k' ),
            color = df2.groupby(by='customer_id').mean()['balance'].sort_values(ascending=False).head(10)
          )
fig.update_traces(textposition='outside')
fig.update_layout(
                    title=dict(text = "Top Valuable Customers by AVG Balance",x=0.5,y=0.95),
                    title_font_size=20,
                    showlegend=False,
                    height = 500
                    
                  )
fig.update_traces(marker_coloraxis=None)
fig.show()

In [ ]:
order = ['August','September','October']
df['month']=pd.Categorical(df['month'],order)
g1 = df.groupby(by='month').agg(['mean','sum'])['amount']
g1.columns=['Avg Amount', 'Total Amount']
g1[['Avg Amount','Total Amount']]=g1[['Avg Amount','Total Amount']].round().astype(int)
g1.reset_index(inplace=True)

g2=df.groupby(by='month').agg(['mean','sum'])['balance']
g2.columns=['Avg Balance', 'Total Balance']
g2[['Avg Balance','Total Balance']]=g2[['Avg Balance','Total Balance']].round().astype(int)
g2.reset_index(inplace=True)

month = g1.merge(g2,on='month')
month

In [ ]:
pio.templates.default = "plotly_white"
fig = ff.create_table(month) 
for i in range(len(fig.layout.annotations)):
    fig.layout.annotations[i].font.size = 13
fig.show()

In [ ]:
# Insights:

# There is a 7% increase in Avg transaction amount from August to October.
# 71% increase in AVG balance maintained by the customers.
# 77% increase in total balance over these 3 months.